# IPL 2026 — Data Integration and Preparation

## 1. Project Setup and Data Loading

### Objective
Load the original IPL 2026 datasets, establish reusable project paths, and inspect the datasets before integration.

### Libraries
- **Pandas:** Data loading, manipulation, and integration.
- **NumPy:** Numerical operations.
- **OS and Pathlib:** File and directory management.

In [1]:
# Step 1: Import libraries and configure paths

import pandas as pd
import numpy as np
from pathlib import Path

# Project directories
PROJECT_DIR = Path.cwd().parent
ORIGINAL_DATA = PROJECT_DIR / "Original-Data"
CLEANED_DATA = PROJECT_DIR / "Cleaned-Data"

# Create the output directory if it doesn't exist
CLEANED_DATA.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Original data directory:", ORIGINAL_DATA)
print("Cleaned data directory:", CLEANED_DATA)

Project directory: c:\Users\Mohit shrivastav\OneDrive\Desktop\ipl-2026-analytic
Original data directory: c:\Users\Mohit shrivastav\OneDrive\Desktop\ipl-2026-analytic\Original-Data
Cleaned data directory: c:\Users\Mohit shrivastav\OneDrive\Desktop\ipl-2026-analytic\Cleaned-Data


# Load the datasets

In [2]:
# Step 2: Load original datasets

matches = pd.read_csv(ORIGINAL_DATA / "matches.csv")
deliveries = pd.read_csv(ORIGINAL_DATA / "deliveries.csv")
batting_stats = pd.read_csv(ORIGINAL_DATA / "batting_stats.csv")
bowling_stats = pd.read_csv(ORIGINAL_DATA / "bowling_stats.csv")
fielding_stats = pd.read_csv(ORIGINAL_DATA / "fielding_stats.csv")
points_table = pd.read_csv(ORIGINAL_DATA / "points_table.csv")
squads = pd.read_csv(ORIGINAL_DATA / "squads.csv")

# Store datasets in a dictionary for convenient inspection
datasets = {
    "Matches": matches,
    "Deliveries": deliveries,
    "Batting": batting_stats,
    "Bowling": bowling_stats,
    "Fielding": fielding_stats,
    "Points Table": points_table,
    "Squads": squads
}

# Display dataset dimensions
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

Matches: 39 rows × 23 columns
Deliveries: 17477 rows × 19 columns
Batting: 15 rows × 17 columns
Bowling: 15 rows × 17 columns
Fielding: 15 rows × 6 columns
Points Table: 10 rows × 9 columns
Squads: 257 rows × 6 columns


## 2. Understanding Dataset Relationships

Before integrating datasets, we need to identify common columns and possible join keys. This helps us choose suitable datasets for merging and avoid incorrect joins or duplicated records.

In [3]:
# Step 2: Inspect columns and sample records

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(name.upper())
    print("Columns:", df.columns.tolist())
    display(df.head(3))


MATCHES
Columns: ['match_id', 'date', 'venue', 'team1', 'team2', 'stage', 'toss_winner', 'toss_decision', 'first_ings_score', 'first_ings_wkts', 'second_ings_score', 'second_ings_wkts', 'match_result', 'match_winner', 'wb_runs', 'wb_wickets', 'balls_left', 'player_of_the_match', 'top_scorer', 'highscore', 'best_bowling', 'best_bowling_figure', 'super_over_match']


,match_id,date,venue,team1,team2,stage,toss_winner,toss_decision,first_ings_score,first_ings_wkts,...,match_winner,wb_runs,wb_wickets,balls_left,player_of_the_match,top_scorer,highscore,best_bowling,best_bowling_figure,super_over_match
0,1,"March 28, 2026","M. Chinnaswamy Stadium, Bangalore",RCB,SRH,League,RCB,Bowl,201.0,9.0,...,RCB,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No
1,2,"March 29, 2026","Wankhede Stadium, Mumbai",MI,KKR,League,MI,Bowl,220.0,4.0,...,MI,NaN,6.0,5.0,Shardul Thakur,Ryan Rickelton,81.0,Shardul Thakur,3--39,No
2,3,"March 30, 2026","Barsapara Stadium, Guwahati",RR,CSK,League,RR,Bowl,127.0,10.0,...,RR,NaN,8.0,47.0,Nandre Burger,Vaibhav Sooryavanshi,52.0,Jofra Archer,2--19,No



DELIVERIES
Columns: ['match_no', 'date', 'stage', 'venue', 'batting_team', 'bowling_team', 'innings', 'over', 'striker', 'bowler', 'runs_of_bat', 'extras', 'wide', 'legbyes', 'byes', 'noballs', 'wicket_type', 'player_dismissed', 'fielder']


,match_no,date,stage,venue,batting_team,bowling_team,innings,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs,wicket_type,player_dismissed,fielder
0,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.1,Travis Head,Jacob Duffy,0,0,0,0,0,0,NaN,NaN,NaN
1,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.2,Travis Head,Jacob Duffy,1,0,0,0,0,0,NaN,NaN,NaN
2,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.3,Abhishek Sharma,Jacob Duffy,0,0,0,0,0,0,NaN,NaN,NaN



BATTING
Columns: ['position', 'batsman', 'team', 'matches', 'innings', 'runs', 'impact', 'average', 'strike_rate', 'not_outs', 'high_score', 'balls_faced', 'hundreds', 'fifties', 'ducks', 'fours', 'sixes']


...

,rank,player,team,matches,catches,cpm
0,1,Rinku Singh,KKR,8,9,1.285
1,2,Devdutt Padikkal,RCB,8,9,1.125
2,3,Dewald Brevis,CSK,5,8,1.600



POINTS TABLE
Columns: ['position', 'team', 'matches', 'wins', 'defeats', 'ties', 'abandoned', 'points', 'nrr']


,position,team,matches,wins,defeats,ties,abandoned,points,nrr
0,1,Punjab Kings,7,6,0,0,1,13,1.333
1,2,Royal Challengers Bengaluru,8,6,2,0,0,12,1.919
2,3,Sunrisers Hyderabad,8,5,3,0,0,10,0.815



SQUADS
Columns: ['team_no', 'team_name', 'player', 'nationality', 'role', 'designation']


,team_no,team_name,player,nationality,role,designation
0,1,Mumbai Indians,Hardik Pandya,Indian,All Rounder,Captain
1,1,Mumbai Indians,Rohit Sharma,Indian,Batter,NaN
2,1,Mumbai Indians,Suryakumar Yadav,Indian,Batter,NaN


## 3. Match and Delivery Relationship

The matches dataset contains match-level information, while the deliveries dataset contains ball-by-ball records.

Before integrating them, we will compare their match identifiers to identify matching records and unmatched IDs. This ensures that the join does not silently exclude or misrepresent matches.

In [4]:
# Step 3: Compare match identifiers

match_ids = set(matches["match_id"].dropna().unique())
delivery_ids = set(deliveries["match_no"].dropna().unique())

print("Unique match IDs in matches:", len(match_ids))
print("Unique match IDs in deliveries:", len(delivery_ids))

print("\nCommon match IDs:", len(match_ids & delivery_ids))
print("Only in matches:", len(match_ids - delivery_ids))
print("Only in deliveries:", len(delivery_ids - match_ids))

print("\nIDs only in matches:")
print(sorted(match_ids - delivery_ids))

print("\nIDs only in deliveries:")
print(sorted(delivery_ids - match_ids))

Unique match IDs in matches: 39
Unique match IDs in deliveries: 74

Common match IDs: 39
Only in matches: 0
Only in deliveries: 35

IDs only in matches:
[]

IDs only in deliveries:
[np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(74)]


## 4. Integrating Match and Delivery Data

We use a left join to combine match-level metadata with ball-by-ball delivery records.

The deliveries dataset is the left table, ensuring that all delivery records are preserved. Match IDs without corresponding metadata will have missing values in the newly added match columns.

The original datasets remain unchanged.

In [5]:
# Step 4: Merge matches with deliveries

match_delivery = deliveries.merge(
    matches,
    left_on="match_no",
    right_on="match_id",
    how="left",
    suffixes=("_delivery", "_match"),
    indicator=True
)

print("Original deliveries shape:", deliveries.shape)
print("Integrated dataset shape:", match_delivery.shape)

print("\nMerge status:")
display(match_delivery["_merge"].value_counts())

print("\nUnmatched delivery records:")
print(
    match_delivery.loc[
        match_delivery["_merge"] == "left_only",
        "match_no"
    ].nunique()
)

print("\nSample integrated records:")
display(match_delivery.head())

Original deliveries shape: (17477, 19)
Integrated dataset shape: (17477, 43)

Merge status:


_merge
both          9048
left_only     8429
right_only       0
Name: count, dtype: int64


Unmatched delivery records:
35

Sample integrated records:


,match_no,date_delivery,stage_delivery,venue_delivery,batting_team,bowling_team,innings,over,striker,bowler,...,wb_runs,wb_wickets,balls_left,player_of_the_match,top_scorer,highscore,best_bowling,best_bowling_figure,super_over_match,_merge
0,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.1,Travis Head,Jacob Duffy,...,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No,both
1,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.2,Travis Head,Jacob Duffy,...,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No,both
2,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.3,Abhishek Sharma,Jacob Duffy,...,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No,both
3,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.4,Abhishek Sharma,Jacob Duffy,...,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No,both
4,1,"Mar 28, 2026",League stage,"M.Chinnaswamy Stadium, Bengaluru",SRH,RCB,1,0.5,Abhishek Sharma,Jacob Duffy,...,NaN,6.0,26.0,Jacob Duffy,Ishan Kishan,80.0,Jacob Duffy,3--22,No,both


## Step 5: Merge Validation

In this step, we validate the integrated dataset by checking:

- Whether the merge changed the total number of delivery records.
- How many delivery records matched with the match metadata.
- How many records remain unmatched.
- Whether any match metadata is missing after the merge.

In [6]:
print("Original deliveries:", len(deliveries))
print("Merged records:", len(match_delivery))

print("\nMerge status (delivery rows):")
print(match_delivery["_merge"].value_counts())

print("\nUnique unmatched match IDs:")
print(
    match_delivery.loc[
        match_delivery["_merge"] == "left_only",
        "match_no"
    ].nunique()
)

print("\nMissing match metadata:")
metadata_cols = [
    "match_id",
    "match_winner",
    "player_of_the_match"
]
print(match_delivery[metadata_cols].isna().sum())

Original deliveries: 17477
Merged records: 17477

Merge status (delivery rows):
_merge
both          9048
left_only     8429
right_only       0
Name: count, dtype: int64

Unique unmatched match IDs:
35

Missing match metadata:
match_id               8429
match_winner           8451
player_of_the_match    8451
dtype: int64


## Step 6: Investigating Missing Match Metadata

We will check whether some records matched with the match table but still have missing values in important match-level columns.

This helps distinguish missing source data from records that failed to match.

In [7]:
matched_rows = match_delivery[
    match_delivery["_merge"] == "both"
]

print("Matched delivery rows:", len(matched_rows))

print("\nMissing values among matched rows:")
print(
    matched_rows[
        ["match_id", "match_winner", "player_of_the_match"]
    ].isna().sum()
)

print("\nMatch IDs with missing winner:")
display(
    matched_rows.loc[
        matched_rows["match_winner"].isna(),
        ["match_no", "match_id"]
    ].drop_duplicates()
)

Matched delivery rows: 9048

Missing values among matched rows:
match_id                0
match_winner           22
player_of_the_match    22
dtype: int64

Match IDs with missing winner:


,match_no,match_id
2612,12,12.0


## Step 7: Inspecting Match 12

We will inspect the original match metadata for Match 12 to understand why its winner and player-of-the-match fields are missing.

We will not fill or modify these values without verifying the source.

In [8]:
display(
    matches.loc[
        matches["match_id"] == 12
    ].T
)

,11
match_id,12
date,"April 06, 2026"
venue,"Eden Gardens, Kolkata"
team1,KKR
team2,PBKS
stage,League
toss_winner,KKR
toss_decision,Bat
first_ings_score,NaN
first_ings_wkts,NaN


## Integration Findings

- The deliveries dataset contains 17,477 records.
- A left merge with the matches dataset preserved all 17,477 records.
- 9,048 delivery rows matched with match metadata.
- 8,429 delivery rows remained unmatched, corresponding to 35 distinct match IDs.
- Match 12 was confirmed as abandoned. Its missing winner, player-of-the-match, and innings statistics are retained as missing values.
- No rows were removed or duplicated during integration.
- The unmatched match IDs remain a known source-data limitation and require consideration during further analysis.

**Decision:** Preserve unmatched delivery records and legitimate missing values. Do not invent or automatically fill match-level information.

## Step 8: Investigating Unmatched Match IDs

The previous merge validation identified 35 match IDs in the deliveries dataset that do not have corresponding records in the matches dataset.

We will inspect these IDs to understand the extent of the missing match metadata before deciding how to handle them.

In [9]:
# Find match IDs that have delivery records but no match metadata
unmatched_ids = sorted(
    set(deliveries["match_no"].dropna().unique()) -
    set(matches["match_id"].dropna().unique())
)

print("Number of unmatched match IDs:", len(unmatched_ids))
print("\nUnmatched match IDs:")
print(unmatched_ids)

# Count delivery records for each unmatched match
unmatched_summary = (
    deliveries[deliveries["match_no"].isin(unmatched_ids)]
    .groupby("match_no")
    .agg(
        delivery_records=("match_no", "size"),
        innings_count=("innings", "nunique"),
        batting_teams=("batting_team", "nunique"),
        bowling_teams=("bowling_team", "nunique")
    )
    .reset_index()
)

display(unmatched_summary)

Number of unmatched match IDs: 35

Unmatched match IDs:
[np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), np.int64(73), np.int64(74)]


,match_no,delivery_records,innings_count,batting_teams,bowling_teams
0,40,242,2,2,2
1,41,243,2,2,2
2,42,217,2,2,2
3,43,248,2,2,2
4,44,234,2,2,2
5,45,235,2,2,2
6,46,248,2,2,2
7,47,246,2,2,2
8,48,229,2,2,2
9,49,253,2,2,2


## Step 9: Inspecting Available Details for Unmatched Matches

The delivery dataset contains dates, venues, and team information for unmatched match IDs.

We will summarize these fields to understand which match-level details can be recovered directly from the delivery records.

In [10]:
unmatched_details = (
    deliveries[deliveries["match_no"].isin(unmatched_ids)]
    .groupby("match_no")
    .agg(
        date=("date", "first"),
        venue=("venue", "first"),
        batting_teams=("batting_team", lambda x: ", ".join(sorted(x.dropna().unique()))),
        bowling_teams=("bowling_team", lambda x: ", ".join(sorted(x.dropna().unique())))
    )
    .reset_index()
)

display(unmatched_details)

,match_no,date,venue,batting_teams,bowling_teams
0,40,"Apr 28, 2026",Maharaja Yadavindra Singh International Cricke...,"PBKS, RR","PBKS, RR"
1,41,"Apr 29, 2026","Wankhede Stadium, Mumbai","MI, SRH","MI, SRH"
2,42,"Apr 30, 2026","Narendra Modi Stadium, Ahmedabad","GT, RCB","GT, RCB"
3,43,"May 1, 2026","Sawai Mansingh Stadium, Jaipur","DC, RR","DC, RR"
4,44,"May 2, 2026","MA Chidambaram Stadium, Chennai","CSK, MI","CSK, MI"
5,45,"May 3, 2026","Rajiv Gandhi International Stadium, Hyderabad","KKR, SRH","KKR, SRH"
6,46,"May 3, 2026","Narendra Modi Stadium, Ahmedabad","GT, PBKS","GT, PBKS"
7,47,"May 4, 2026","Wankhede Stadium, Mumbai","LSG, MI","LSG, MI"
8,48,"May 5, 2026","Arun Jaitley Stadium, Delhi","CSK, DC","CSK, DC"
9,49,"May 6, 2026","Rajiv Gandhi International Stadium, Hyderabad","PBKS, SRH","PBKS, SRH"


## Step 10: Final Integration Checks

We will check the availability of key delivery-level fields and confirm that the integrated dataset retains all delivery records.

The original match metadata will remain available wherever a matching record exists. Missing metadata will be preserved rather than filled with unverified values.

In [11]:
# Check important columns in the integrated dataset
key_columns = [
    "match_no",
    "date_delivery",
    "venue_delivery",
    "batting_team",
    "bowling_team",
    "innings",
    "striker",
    "bowler",
    "runs_of_bat",
    "extras",
    "match_id",
    "match_winner"
]

print("Integrated dataset shape:", match_delivery.shape)

print("\nMissing values in key columns:")
display(match_delivery[key_columns].isna().sum())

print("\nUnique matches in integrated dataset:",
      match_delivery["match_no"].nunique())

print("\nDuplicate rows in integrated dataset:",
      match_delivery.duplicated().sum())

Integrated dataset shape: (17477, 43)

Missing values in key columns:


match_no             0
date_delivery        0
venue_delivery       0
batting_team         0
bowling_team         0
innings              0
striker              0
bowler               0
runs_of_bat          0
extras               0
match_id          8429
match_winner      8451
dtype: int64


Unique matches in integrated dataset: 74

Duplicate rows in integrated dataset: 0


## Step 11: Saving the Integrated Dataset

We will export the integrated delivery and match metadata into a CSV file.

The original source files remain unchanged. Unmatched records and legitimate missing values are preserved.

In [12]:
# Remove the merge indicator from the exported dataset
integrated_data = match_delivery.drop(columns=["_merge"])

# Save the integrated dataset
output_path = CLEANED_DATA / "ipl_2026_integrated.csv"
integrated_data.to_csv(output_path, index=False)

print("Integrated dataset saved successfully!")
print("File:", output_path)
print("Shape:", integrated_data.shape)

Integrated dataset saved successfully!
File: c:\Users\Mohit shrivastav\OneDrive\Desktop\ipl-2026-analytic\Cleaned-Data\ipl_2026_integrated.csv
Shape: (17477, 42)
